# Dual-system VLA from first principles

**Paper:** C. Cui, P. Ding, W. Song, et al., "OpenHelix: A Short Survey, Empirical Analysis, and Open-Source
Dual-System VLA Model for Robotic Manipulation," 2025. [arXiv:2505.03912](https://arxiv.org/abs/2505.03912)

**Variants covered:** LCB ([arXiv:2405.04798](https://arxiv.org/abs/2405.04798)), DP-VLA (CoRL 2024), HiRT
([arXiv:2410.05273](https://arxiv.org/abs/2410.05273)), RoboDual ([arXiv:2410.08001](https://arxiv.org/abs/2410.08001)),
DexVLA ([arXiv:2502.05855](https://arxiv.org/abs/2502.05855)), OpenHelix.

**Implementation:** `src/Halo_VLA/dual_system/` · docs: `docs/dual_system.md` · selected with
`--dual_system none | basic | lcb | dp_vla | hirt | robodual | dexvla | openhelix | custom`.

| § | content |
|---|---|
| 1 | the problem and dual-process theory |
| 2 | the basic dual system, built piece by piece |
| 3 | a toy task where the two systems must cooperate, trained end to end |
| 4 | ablations: what each system contributes |
| 5 | asynchrony: latency vs. latent staleness |
| 6 | every published variant: diagram, configuration, trainable parameters |

## 1. The problem

A vision-language model generalises across instructions and scenes but is slow (RT-2: 1-5 Hz). A small visuomotor
policy is fast but brittle. **Dual-process theory** splits the work:

* **System 2** (slow, deliberate): the VLM reads the instruction + scene and writes a *latent*. Updated rarely.
* **System 1** (fast, reactive): a small policy reads the *current* frame + proprioception at control rate and acts,
  conditioned on the latest - possibly stale - latent.

For this to be a real dual system, System 1 must perceive the scene itself, and the two systems must run at different rates.

In [1]:
import sys, time
from pathlib import Path
for p in (Path.cwd().parents[1], Path.cwd().parents[1] / "src" / "Halo_VLA"):
    sys.path.insert(0, str(p))

import torch
import matplotlib.pyplot as plt
from config import HaloVLMConfig
from models.halo_vla import HaloVLM
from dual_system import PRESETS, DualSystemController, DualSystemVLA, resolve_dual_system_config

torch.manual_seed(0)
print("presets:", ["none", *PRESETS, "custom"])

presets: ['none', 'basic', 'lcb', 'dp_vla', 'hirt', 'robodual', 'dexvla', 'openhelix', 'custom']


## 2. The basic dual system

```mermaid
flowchart LR
  subgraph S2["System 2 · slow (every async_steps)"]
    direction TB
    IMG2["camera frames<br/>(possibly stale)"]:::data --> VLM["HaloVLM trunk<br/>ViT + projector + decoder"]:::slow
    TXT["instruction"]:::data --> VLM
    ACT["learnable &lt;ACT&gt; query"]:::learn --> VLM
    VLM --> Z["latent z = final state of &lt;ACT&gt;"]:::latent
  end
  Z --> PROJ["projector"]:::learn
  subgraph S1["System 1 · fast (every control step)"]
    direction TB
    IMG1["current frame"]:::data --> ENC["fast conv encoder"]:::fast
    PROP["proprioception"]:::data --> PE["MLP"]:::fast
    PROJ --> XATT
    ENC --> XATT["action transformer<br/>cross-attends to latent + obs + proprio"]:::fast
    PE --> XATT
    XATT --> HEAD["flow-matching head"]:::fast
  end
  HEAD --> A["action chunk"]:::out
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef slow fill:#e8eefc,stroke:#3b5bdb,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

The pieces, from the slow side to the fast side:

1. **System 2** = Halo-VLA's VLM trunk (`HaloVLM.encode_sequence`) with a learnable **`<ACT>` query** appended after
   the instruction. In a causal decoder the last token attends to everything, so its final state summarises
   image + instruction: that is the latent `z`.
2. **Projector**: `z` -> System 1's width.
3. **System 1**: a cheap conv encoder of the current frame, an MLP on proprioception, and an action transformer that
   cross-attends to `[z, frame tokens, proprio]` and outputs an action chunk with a flow-matching head.

In [2]:
cfg = HaloVLMConfig(img_size=32, patch_size=16, emb_dim=64, vit_num_layers=1, vit_num_heads=4, vit_mlp_dim=64,
                    dec_num_layers=2, dec_num_heads=4, dec_mlp_dim=64, use_moe=False, enable_visual_dit=False,
                    action_dim=2, state_dim=3, action_chunk_size=2, flow_hidden_dim=32)
small = {"system1_dim": 64, "system1_layers": 2, "system1_heads": 4, "sample_steps": 8}

model = DualSystemVLA(HaloVLM(cfg), resolve_dual_system_config("basic", small))
for part in ("system2", "latent", "bridge", "system1"):
    module = getattr(model, part)
    n = sum(p.numel() for p in module.parameters())
    print(f"{part:8s} {type(module).__name__:22s} {n:>9,} params")
print("<ACT> queries:", tuple(model.system2.act_queries.shape))

system2  System2                6,724,148 params
latent   ActTokenLatent                 0 params
bridge   LatentBridge               8,320 params
system1  System1Policy            173,666 params
<ACT> queries: (1, 64)


## 3. A toy task that needs both systems

Each sample has an instruction word (`w5` = "go up", `w6` = "go down") and two camera frames: an **old** one
and the **current** one. The target action is

* `a[0] = +1 / -1` from the **instruction** (only System 2 reads language),
* `a[1] = 2 * red(current frame) - 1`, the current frame's red level (System 1 sees it; System 2 sees only the old frame,
  because it lags at deployment).

In [3]:
IMG, ACT, STATE = cfg.image_token_id, cfg.action_token_id, cfg.state_token_id

def make_batch(n, seed=None):
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    word = torch.randint(5, 7, (n,), generator=g)                       # 5 = up, 6 = down
    red_old, red_now = torch.rand(n, generator=g), torch.rand(n, generator=g)
    frames = torch.rand(n, 2, 3, 32, 32, generator=g) * 0.1
    frames[:, 0, 0] += red_old.view(n, 1, 1); frames[:, 1, 0] += red_now.view(n, 1, 1)
    ids = torch.stack([torch.tensor([IMG, IMG, 7, int(w), 8, STATE, ACT]) for w in word])
    target = torch.stack([torch.where(word == 5, 1.0, -1.0), 2 * red_now - 1], -1)
    return {"images": frames, "image_mask": torch.ones(n, 2), "input_ids": ids, "attention_mask": torch.ones_like(ids),
            "labels": ids.clone(), "states": torch.zeros(n, 1, 3),
            "actions": target.unsqueeze(1).repeat(1, 2, 1), "action_mask": torch.ones(n, 2)}

b = make_batch(4, seed=0)
print({k: tuple(v.shape) for k, v in b.items()})
print("targets:", b["actions"][:, 0].round(decimals=2).tolist())

{'images': (4, 2, 3, 32, 32), 'image_mask': (4, 2), 'input_ids': (4, 7), 'attention_mask': (4, 7), 'labels': (4, 7), 'states': (4, 1, 3), 'actions': (4, 2, 2), 'action_mask': (4, 2)}
targets: [[1.0, -0.09000000357627869], [-1.0, 0.25999999046325684], [-1.0, -0.30000001192092896], [1.0, -0.20000000298023224]]


In [4]:
torch.manual_seed(0)
model = DualSystemVLA(HaloVLM(cfg), resolve_dual_system_config("basic", {**small, "system2_frame_delay": 1})).train()
opt = torch.optim.AdamW(model.optimizer_parameters(), lr=2e-3)
history = []
for step in range(300):
    loss = model(make_batch(32)).total
    opt.zero_grad(); loss.backward(); opt.step()
    history.append(loss.item())

plt.figure(figsize=(6, 3)); plt.plot(torch.tensor(history).view(-1, 10).mean(1)); plt.yscale("log")
plt.xlabel("step / 10"); plt.ylabel("flow-matching loss"); plt.title("training the basic dual system"); plt.show()
print(f"loss {history[0]:.3f} -> {sum(history[-20:]) / 20:.3f}")

loss 2.238 -> 0.152


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_20830/2981860681.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("step / 10"); plt.ylabel("flow-matching loss"); plt.title("training the basic dual system"); plt.show()


`system2_frame_delay=1` trains System 2 on the **old** frame (HiRT-style latency simulation), so the colour signal can only
come through System 1's own perception.

## 4. What each system contributes

Evaluate the trained model on held-out samples, then break one system at a time:

* **zero latent**: System 1 gets no instruction information;
* **stale frame for System 1**: System 1 sees the old frame instead of the current one.

In [5]:
@torch.no_grad()
def errors(model, batch, latent=None, stale_frame=False):
    batch = dict(batch)
    if stale_frame:
        batch["images"] = batch["images"][:, [0, 0]]
    pred = model.predict_actions(batch, latent=latent)[:, 0]
    err = (pred - batch["actions"][:, 0]).abs().mean(0)
    return [round(e, 3) for e in err.tolist()]

model.eval(); torch.manual_seed(1)
test = make_batch(256, seed=123)
z, z_mask, _ = model.encode_latent(test)
rows = {
    "full dual system": errors(model, test),
    "zero latent (no System 2)": errors(model, test, latent=(torch.zeros_like(z), z_mask)),
    "stale frame for System 1": errors(model, test, stale_frame=True),
}
print(f"{'':28s} |err| instruction dim | |err| colour dim")
for k, (a0, a1) in rows.items():
    print(f"{k:28s} {a0:>20} {a1:>16}")

                             |err| instruction dim | |err| colour dim
full dual system                             0.07            0.095
zero latent (no System 2)                   1.035            0.109
stale frame for System 1                    0.068            0.651


Read the table this way:
* The full system is accurate on both dimensions.
* Removing the latent breaks the **instruction** dimension: only System 2 read the language.
* A stale frame breaks the **colour** dimension: only System 1's own, current perception carries it.

That is the division of labour the architecture is designed for.

## 5. Asynchrony: latency vs. staleness

```mermaid
sequenceDiagram
  participant Env as Robot / camera
  participant S1 as System 1 (fast)
  participant S2 as System 2 (slow)
  Env->>S2: frame t0 + instruction
  S2-->>S1: latent z(t0)
  loop every control step t
    Env->>S1: frame t + proprioception
    S1->>Env: action chunk (uses latest z)
    alt t mod async_steps == 0
      Env->>S2: frame t (inline, or background thread)
      S2-->>S1: latent z(t) replaces z
    end
  end
```

`DualSystemController` runs System 2 every `async_steps` control steps and System 1 every step. In this toy model
System 2 is small, so the gap is modest; with a 7B VLM it is the difference between ~1 Hz and tens of Hz.

In [6]:
@torch.no_grad()
def control_rate(model, async_steps, steps=60):
    ctrl = DualSystemController(model, async_steps=async_steps)
    obs = make_batch(1, seed=7)
    t0 = time.perf_counter()
    for _ in range(steps):
        ctrl.act(obs)
    return steps / (time.perf_counter() - t0), ctrl.system2_calls

for k in (1, 5, 10, 60):
    hz, calls = control_rate(model, k)
    print(f"async_steps={k:>2}: {hz:7.1f} control steps/s   System 2 calls: {calls}")

async_steps= 1:   247.0 control steps/s   System 2 calls: 60
async_steps= 5:   326.5 control steps/s   System 2 calls: 12


async_steps=10:   337.4 control steps/s   System 2 calls: 6
async_steps=60:   350.4 control steps/s   System 2 calls: 1


In [7]:
# Staleness: refresh the latent only every k steps while the instruction stays fixed.
# The latent here encodes the instruction (as OpenHelix found for <ACT> latents), so staleness costs little.
test_small = make_batch(64, seed=5)
z, z_mask, _ = model.encode_latent(test_small)
old = dict(test_small, images=test_small["images"][:, [0, 0]])
z_old, m_old, _ = model.encode_latent(old)   # a latent computed from an older observation
print("fresh latent :", errors(model, test_small, latent=(z, z_mask)))
print("stale latent :", errors(model, test_small, latent=(z_old, m_old)))

fresh latent : [0.083, 0.107]
stale latent : [0.075, 0.111]


## 6. The published variants

Each is the same parts with different answers to the design questions.

### LCB - `--dual_system lcb`

1 `<ACT>` latent, fine-tuned System 2, diffusion actor, alignment loss keeping the latent close to the instruction.

```mermaid
flowchart LR
  I["frames + instruction + &lt;ACT&gt;"]:::data --> V["System 2<br/>finetune / LoRA"]:::slow --> Z["z = &lt;ACT&gt; state"]:::latent
  Z --> P["MLP projector"]:::learn --> D["diffusion actor<br/>(DDPM, DDIM sampling)"]:::fast
  O["current frame + proprio"]:::data --> D --> A["action chunk"]:::out
  Z -.-> L["alignment loss<br/>cos(z, instruction embedding)"]:::loss
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef slow fill:#e8eefc,stroke:#3b5bdb,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef loss fill:#f8f9fa,stroke:#868e96,color:#222,stroke-dasharray:4 3
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### DP-VLA - `--dual_system dp_vla`

frozen System 2, all last-layer states as the latent, regression transformer trained from scratch.

```mermaid
flowchart LR
  I["frames + instruction"]:::data --> V["System 2<br/>frozen"]:::frozen --> H["all last-layer states"]:::latent
  H --> P["MLP projector"]:::learn --> T["transformer policy<br/>regression, from scratch"]:::fast
  O["current frame + proprio"]:::data --> T --> A["action chunk"]:::out
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef frozen fill:#eceff3,stroke:#495057,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### HiRT - `--dual_system hirt`

MAP-pooled latent, delayed System 2 view during training, regression policy.

```mermaid
flowchart LR
  F["frame t-1 (delayed view)"]:::data --> V["System 2<br/>finetune / LoRA"]:::slow
  I["instruction"]:::data --> V
  V --> M["MAP pooling<br/>learned probe attends to all states"]:::learn --> Z["1 pooled token"]:::latent
  Z --> P["MLP projector"]:::learn --> R["RT-1-style policy<br/>regression"]:::fast
  O["frame t + proprio"]:::data --> R --> A["action chunk"]:::out
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef slow fill:#e8eefc,stroke:#3b5bdb,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### RoboDual - `--dual_system robodual`

4 `<ACT>` (action latent) + instruction states (task latent), DiT diffusion, Perceiver-resampled observations.

```mermaid
flowchart LR
  I["frames + instruction + 4 x &lt;ACT&gt;"]:::data --> V["System 2<br/>finetune / LoRA"]:::slow
  V --> ZA["action latent<br/>4 &lt;ACT&gt; states"]:::latent
  V --> ZT["task latent<br/>instruction states"]:::latent
  ZA --> P["MLP projector"]:::learn
  ZT --> P
  O["current frame"]:::data --> E["fast encoder"]:::fast --> R["Perceiver resampler<br/>16 tokens"]:::learn
  P --> D["DiT diffusion policy"]:::fast
  R --> D
  S["proprio"]:::data --> D
  D --> A["action chunk"]:::out
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef slow fill:#e8eefc,stroke:#3b5bdb,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### DexVLA - `--dual_system dexvla`

instruction states as the latent, diffusion expert.

```mermaid
flowchart LR
  I["frames + instruction"]:::data --> V["System 2<br/>finetune / LoRA"]:::slow --> L["instruction-token states"]:::latent
  L --> P["MLP projector"]:::learn --> D["scaled diffusion policy"]:::fast
  O["current frame + proprio"]:::data --> D --> A["action chunk"]:::out
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef slow fill:#e8eefc,stroke:#3b5bdb,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### OpenHelix - `--dual_system openhelix`

frozen System 2 with a prompt-tuned `<ACT>`, auxiliary action head, two-stage pre-alignment.

```mermaid
flowchart LR
  I["frames + instruction + &lt;ACT&gt;"]:::data --> V["System 2<br/>frozen"]:::frozen
  Q["&lt;ACT&gt; embedding<br/>the only System 2 weights that learn"]:::learn --> V
  V --> Z["z = &lt;ACT&gt; state"]:::latent
  Z --> P["linear projector"]:::learn --> D["diffusion policy<br/>stage 1: held fixed · stage 2: trained"]:::fast
  O["current frame + proprio"]:::data --> D --> A["action chunk"]:::out
  Z --> X["auxiliary MLP<br/>predicts the action chunk"]:::learn -.-> L["auxiliary L1 loss"]:::loss
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef frozen fill:#eceff3,stroke:#495057,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef loss fill:#f8f9fa,stroke:#868e96,color:#222,stroke-dasharray:4 3
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### Middle-layer latent - `--dual_system custom --dual_system_option latent=middle_layer`

```mermaid
flowchart LR
  I["frames + instruction"]:::data --> L1["decoder layers 0..k"]:::slow --> H["layer k states"]:::latent
  L1 --> L2["layers k+1..N<br/>(not needed for the latent)"]:::frozen
  H --> P["MLP projector"]:::learn --> D["System 1 policy"]:::fast --> A["action chunk"]:::out
  O["current frame + proprio"]:::data --> D
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef slow fill:#e8eefc,stroke:#3b5bdb,color:#222
  classDef frozen fill:#eceff3,stroke:#495057,color:#222
  classDef fast fill:#e9f7ef,stroke:#2f9e44,color:#222
  classDef learn fill:#f3e8fd,stroke:#8e44ad,color:#222
  classDef latent fill:#fff0f6,stroke:#c2255c,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

In [8]:
rows = []
for name in [*PRESETS, "custom"]:
    over = {**small, "latent": "middle_layer"} if name == "custom" else small
    torch.manual_seed(0)
    m = DualSystemVLA(HaloVLM(cfg), resolve_dual_system_config(name, over)).train()
    losses = m(make_batch(8, seed=0)); losses.total.backward()
    c = m.config
    trainable = sum(p.numel() for p in m.optimizer_parameters())
    s2 = sum(p.numel() for n, p in m.named_parameters() if p.requires_grad and n.startswith("system2."))
    rows.append((name, c.latent, c.system2_training, c.policy, trainable, s2, losses.total.item()))
print(f"{'preset':10s} {'latent':13s} {'System 2':14s} {'policy':14s} {'trainable':>10s} {'of which S2':>12s} {'loss':>7s}")
for r in rows:
    print(f"{r[0]:10s} {r[1]:13s} {r[2]:14s} {r[3]:14s} {r[4]:>10,} {r[5]:>12,} {r[6]:7.3f}")

preset     latent        System 2       policy          trainable  of which S2    loss
basic      act_token     finetune       flow_matching   6,897,586    6,715,600   1.963
lcb        act_token     finetune       diffusion       6,897,586    6,715,600   1.869
dp_vla     last_hidden   frozen         regression        181,986            0   0.514
hirt       pooled        finetune       regression      6,914,354    6,715,536   1.339
robodual   multi_act     finetune       diffusion       6,948,786    6,715,792   1.065
dexvla     language      finetune       diffusion       6,897,522    6,715,536   2.070
openhelix  act_token     prompt_tuning  diffusion         182,310           64   2.714
custom     middle_layer  finetune       flow_matching   6,897,522    6,715,536   1.635


## Recap

* A dual system = slow System 2 (VLM) + latent bridge + fast System 1 (policy with its own perception), running at two
  rates.
* The published designs differ in **what** crosses the bridge, **who** trains, and **how** the latent is kept useful.
  Here each one is a preset of the same code.
* Without the latent, the instruction is lost; without System 1's own current view, the scene's state is lost.
  Asynchrony saves System 2 compute, and costs little when the latent mostly carries the instruction.

**Exercises**

1. Train with `system2_frame_delay=0`, then evaluate the "stale frame" ablation. Does System 2 now carry colour?
2. Replace `basic` with `openhelix` (prompt tuning). Can a frozen System 2 still route the instruction?
3. Compare `policy=regression` and `policy=diffusion` on the toy task with the same budget.